## MFI Value Test — Multi-Prompt Comparison

Tests whether MFI data (with baseline prompt) consistently improves insights over property counts alone across diverse infrastructure prompts.

Both approaches use the same system prompt. The difference is:
- **Baseline**: payload contains only `resourceContext` (property counts)
- **MFI**: payload contains `resourceContext` + `tenantPatterns` (FP-Growth MFIs)

Data processing runs once. Each prompt generates 2 insight sets (baseline + MFI) and 2 evaluations (both orderings).

In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from openai import OpenAI

# -------- Prompts to test --------
PROMPTS = [
    "Internal web app for the finance team with a relational database backend",
    "Deploy a geo-redundant backup solution for on-premises SQL servers using Azure Backup, configure encryption-at-rest, and automate monthly DR tests.",
    "Deploy 3-tier architecture with hardened OS images, VM backups scheduled daily, and application-level redundancy for the business logic tier.",
    "Configure a site recovery plan for disaster failover from East to West Azure region, replicate major VM workloads, and automate DNS failbacks.",
    "Provision a jumpbox VM for secure management, establish NSGs for each tier, and connect tiers using internal Azure Load Balancer.",
    "Spin up Linux VMs for each tier using Terraform, automate patch management via Azure Automation, and log traffic between subnets for compliance.",
    "Deploy three distinct VM scale sets for a legacy app, route incoming HTTP/S via Application Gateway with WAF, and encrypt all data disks.",
    "Set up Azure Backup for critical VM workloads, create a long-term retention policy for compliance, and test backup restores quarterly.",
    "Deploy disaster recovery for VMware VMs using Azure Site Recovery, configure runbooks for smooth failover, and maintain compliance audit trails.",
]

# -------- Paths --------
ARG_CACHE_PATH = Path("../../plugin/skills/azure-enterprise-infra-planner/scripts/arg_raw_output_all.json")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -------- Azure OpenAI settings --------
AOAI_BASE_URL    = os.environ.get("AZURE_OPENAI_BASE_URL", "https://workloads-assistant-aoai.openai.azure.com/openai/v1")
AOAI_DEPLOYMENT  = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5-mini")
AOAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "preview")

# -------- FP-Growth settings --------
MIN_ABSOLUTE_COUNT = 4
MIN_SUPPORT_FLOOR = 0.02
MIN_ITEMSET_LEN = 2
MAX_ITEMSETS = 200
MIN_TRANSACTIONS = 30

# -------- Property aggregation --------
MAX_PROPERTY_DEPTH = 5


In [ ]:
_credential = DefaultAzureCredential()
_token_provider = get_bearer_token_provider(_credential, "https://cognitiveservices.azure.com/.default")

aoai = OpenAI(
    base_url=AOAI_BASE_URL,
    api_key="placeholder",
    default_query={"api-version": AOAI_API_VERSION},
    default_headers={"Authorization": f"Bearer {_token_provider()}"},
)
print(f"Azure OpenAI client ready (deployment={AOAI_DEPLOYMENT}).")


In [ ]:
cache = Path(ARG_CACHE_PATH)
if cache.exists():
    raw = json.loads(cache.read_text(encoding="utf-8"))
    if isinstance(raw, list):
        resources = raw
    elif isinstance(raw, dict) and "data" in raw:
        resources = raw["data"]
    else:
        raise ValueError(f"Unexpected cache format: {type(raw)}")
else:
    raise FileNotFoundError(f"ARG cache not found at {cache}")

print(f"Loaded {len(resources):,} resources from cache.")


## Noise Filtering & Property Aggregation

In [ ]:
DROP_ARM_CHILD_TYPES = True

AUTO_CREATED_TYPES = frozenset({
    "microsoft.alertsmanagement/smartdetectoralertrules",
    "microsoft.insights/actiongroups",
    "microsoft.alertsmanagement/prometheusrulegroups",
    "microsoft.security/automations",
    "microsoft.security/pricings",
    "microsoft.operationsmanagement/solutions",
    "microsoft.security/iotsecuritysolutions",
    "microsoft.network/networkwatchers",
    "microsoft.advisor/recommendations",
})

INTERNAL_MS_RP_PREFIXES = (
    "microsoft.portalservices/",
    "microsoft.cloudtest/",
    "microsoft.hydra/",
    "microsoft.swiftlet/",
    "microsoft.compute/swiftlets",
    "microsoft.fairfieldgardens/",
    "microsoft.footprintmonitoring/",
    "microsoft.saashub/",
    "microsoft.visualstudio/",
)

AUTO_MANAGED_SUBRESOURCE_TYPES = frozenset({
    "microsoft.containerregistry/registries/replications",
    "microsoft.containerregistry/registries/webhooks",
    "microsoft.compute/capacityreservationgroups/capacityreservations",
    "microsoft.compute/hostgroups/hosts",
    "microsoft.compute/galleries/images/versions",
    "microsoft.network/networkmanagers/ipampools",
    "microsoft.network/networkmanagers/verifierworkspaces",
})

MARKETPLACE_TYPES = frozenset({
    "microsoft.solutions/applications",
    "microsoft.solutions/appliances",
    "microsoft.saas/resources",
    "microsoft.saashub/cloudservices",
})


def _is_noise(rtype: str) -> bool:
    if not rtype:
        return False
    if rtype in AUTO_CREATED_TYPES or rtype in AUTO_MANAGED_SUBRESOURCE_TYPES or rtype in MARKETPLACE_TYPES:
        return True
    if any(rtype.startswith(p) for p in INTERNAL_MS_RP_PREFIXES):
        return True
    return False


def _is_auto_created(rtype: str) -> bool:
    if not rtype:
        return False
    if DROP_ARM_CHILD_TYPES and rtype.count("/") >= 2:
        return True
    return _is_noise(rtype)


PROPERTY_LEAF_WHITELIST = frozenset({
    "location", "kind",
    "sku", "name", "tier", "family", "capacity", "size",
    "publicnetworkaccess", "restrictoutboundnetworkaccess",
    "publicnetworkaccessforingestion", "publicnetworkaccessforquery",
    "defaultaction", "bypass",
    "disablelocalauth", "enablerbacauthorization",
    "minimumtlsversion", "minimaltlsversion",
    "identity",
    "keysource", "enabledoubleencryption", "enablediskencryption",
    "infrastructureencryption", "requireinfrastructureencryption",
    "zoneredundant", "zoneredundancy", "redundancymode", "replication",
    "platformfaultdomaincount",
    "backupretentionintervalinhours", "backupintervalinminutes",
    "backupstorageredundancy",
    "softdeleteretentionindays", "enablesoftdelete", "enablepurgeprotection",
    "ostype", "hypervgeneration", "licensetype",
    "accesstier", "largefilesharesstate", "allowsharedkeyaccess",
    "enablehttpstrafficonly", "supportshttpstrafficonly",
})

_KEY_DENY_RE = re.compile(r"(secret|password|credential|token|sas|certificate|thumbprint|fingerprint|connection|connstr|admin(istrator)?(user|login|name)|private(ip|key|address)|publicip|ipaddress|fqdn|hostname|host_name|endpoint|url|uri|email|mail|principalid|tenantid|subscriptionid|objectid|clientid|appid|customsubdomain|customdomain|key$|^key|accountkey|accesskey|primarykey|secondarykey|sharedkey)", re.IGNORECASE)

_VALUE_DENY_PATTERNS = [
    re.compile(r"^\d{1,3}(\.\d{1,3}){3}$"),
    re.compile(r"(?i)^[0-9a-f:]{6,}$"),
    re.compile(r"(?i)^[0-9a-f]{8}-([0-9a-f]{4}-){3}[0-9a-f]{12}$"),
    re.compile(r"(?i)^https?://"),
    re.compile(r"^[^@]+@[^@]+\.[^@]+$"),
    re.compile(r"^eyJ[A-Za-z0-9_-]+\."),
    re.compile(r"^[A-Za-z0-9+/]{40,}={0,2}$"),
]


def _is_pii_key(key: str) -> bool:
    return bool(_KEY_DENY_RE.search(key))


def _is_pii_value(val: str) -> bool:
    return any(p.search(val) for p in _VALUE_DENY_PATTERNS)


def walk_properties(node: Any, path: tuple[str, ...] = (), depth: int = 0, max_depth: int = MAX_PROPERTY_DEPTH):
    if depth > max_depth:
        return
    if isinstance(node, dict):
        for k, v in node.items():
            k_lower = str(k).lower()
            if _is_pii_key(k_lower):
                continue
            yield from walk_properties(v, path + (k_lower,), depth + 1, max_depth)
    elif isinstance(node, list):
        for item in node:
            yield from walk_properties(item, path, depth + 1, max_depth)
    else:
        val_str = str(node).strip()
        if val_str and not _is_pii_value(val_str):
            yield (path, val_str)


def _insert_aggregation(tree: dict, path: tuple[str, ...], value: str) -> None:
    if not path:
        return
    leaf_name = path[-1]
    if leaf_name not in PROPERTY_LEAF_WHITELIST:
        return
    node = tree
    for segment in path[:-1]:
        node = node.setdefault(segment, {})
    counter = node.setdefault(leaf_name, Counter())
    if isinstance(counter, Counter):
        counter[value] += 1


def _emit_aggregation(tree: dict, total_count: int = 1) -> dict:
    result = {}
    for key, val in tree.items():
        if isinstance(val, Counter):
            top3 = val.most_common(3)
            total = sum(val.values())
            if total > 0:
                result[key] = {v: round(c / total, 3) for v, c in top3}
        elif isinstance(val, dict):
            sub = _emit_aggregation(val, total_count)
            if sub:
                result[key] = sub
    return result


def aggregate_resources(rows: list[dict]) -> dict[str, dict]:
    grouped: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        rtype = (row.get("type") or "").lower()
        if not rtype or _is_noise(rtype):
            continue
        grouped[rtype].append(row)
    aggregations: dict[str, dict] = {}
    for rtype, type_rows in sorted(grouped.items()):
        tree: dict = {}
        for row in type_rows:
            for field in ("location", "kind"):
                val = row.get(field)
                if val and isinstance(val, str):
                    _insert_aggregation(tree, (field,), val.lower())
            sku = row.get("sku")
            if isinstance(sku, dict):
                for p, v in walk_properties(sku, ("sku",)):
                    _insert_aggregation(tree, p, v)
            identity = row.get("identity")
            if isinstance(identity, dict):
                id_type = identity.get("type")
                if id_type and isinstance(id_type, str):
                    _insert_aggregation(tree, ("identity",), id_type)
            props = row.get("properties")
            if isinstance(props, dict):
                for p, v in walk_properties(props, ("properties",)):
                    _insert_aggregation(tree, p, v)
        prop_agg = _emit_aggregation(tree, len(type_rows))
        aggregations[rtype] = {"totalCount": len(type_rows), "propertyAggregations": prop_agg}
    return aggregations


resource_type_aggregations = aggregate_resources(resources)
print(f"Aggregated {len(resource_type_aggregations)} distinct resource types.")


## FP-Growth Mining

In [ ]:
def build_transactions(rows: list[dict]):
    grouped: dict[tuple[str, str], set[str]] = defaultdict(set)
    for row in rows:
        sub = row.get("subscriptionId")
        rg = (row.get("resourceGroup") or "").lower()
        rtype = (row.get("type") or "").lower()
        if not sub or not rg or not rtype:
            continue
        if _is_auto_created(rtype):
            continue
        grouped[(sub, rg)].add(rtype)
    transactions = [sorted(items) for items in grouped.values() if len(items) >= 2]
    return transactions


def _reliability(count: int) -> str:
    if count >= 30: return "high"
    if count >= 10: return "medium"
    if count >= 5: return "low"
    return "exploratory"


def compute_mfis(itemsets: list[dict]) -> list[dict]:
    sets = [(frozenset(it["itemset"]), it) for it in itemsets]
    sets.sort(key=lambda s: (-len(s[0]), -float(s[1].get("support", 0))))
    kept: list[tuple[frozenset, dict]] = []
    for s, entry in sets:
        if any(s < ks for ks, _ in kept):
            continue
        kept.append((s, entry))
    return [entry for _, entry in kept]


transactions = build_transactions(resources)
n = len(transactions)
mfis: list[dict] = []
min_support = 0.0

if n > 0:
    min_support = max(MIN_SUPPORT_FLOOR, MIN_ABSOLUTE_COUNT / n)
    te = TransactionEncoder()
    te_ary = te.fit_transform(transactions)
    df = pd.DataFrame(te_ary, columns=te.columns_)
    freq_df = fpgrowth(df, min_support=min_support, use_colnames=True)
    freq_df = freq_df[freq_df["itemsets"].apply(len) >= MIN_ITEMSET_LEN]
    if len(freq_df) > 0:
        all_itemsets = []
        for _, row in freq_df.iterrows():
            sup = float(row["support"].item()) if hasattr(row["support"], "item") else float(row["support"])
            count = int(round(sup * n))
            all_itemsets.append({
                "itemset": sorted(row["itemsets"]),
                "support": round(sup, 4),
                "count": count,
                "reliability": _reliability(count),
            })
        mfis = compute_mfis(all_itemsets)
        if len(mfis) > MAX_ITEMSETS:
            mfis = sorted(mfis, key=lambda x: (-x["count"], -len(x["itemset"])))[:MAX_ITEMSETS]

print(f"Transactions: {n}")
print(f"Maximal Frequent Itemsets: {len(mfis)}")


## Shared Prompt & Payload Builders

In [ ]:
INSIGHTS_PROMPT = """# Role and Objective
You are an expert Azure Insight Agent. Your mission is to analyze the user's existing infrastructure data and produce insights that inform downstream infrastructure plan generation.

# Input Data Format

You will receive a JSON object with two sections:

## userQuery
The user's infrastructure request. Focus your insights on patterns most relevant to this request, but draw from all tenant-wide data.

## resourceContext
Per-resource-type property aggregations across the tenant.
- Each key is an ARM resource type (e.g. "microsoft.storage/storageaccounts").
- `totalCount`: how many instances of this type exist in the tenant.
- `propertyAggregations`: nested object where each leaf is a dict of `{value: fraction}`.
  - `fraction` is the share of instances that have that value (0.0\u20131.0). The top 3 values are shown; the implied remainder is `1 - sum(fractions)`.
  - Example: `"location": {"eastus": 0.6, "westus2": 0.3}` means 60% of instances are in eastus, 30% in westus2, and 10% elsewhere.

# Process
1. Analyze the property aggregations to identify architectural conventions (SKU choices, security posture, region preferences, redundancy settings).
2. Identify resource types that are relevant to the user's query and highlight their conventions.
3. Include important tenant-wide conventions even if not directly query-related.
4. Re-examine your insights for completeness and accuracy.

# Insight Guidelines
When selecting resource properties to base insights on:
- Only consider properties that represent explicit user decisions affecting design.
- Never include properties involving runtime, versions, implementation details, app settings, default values, operational settings, or boilerplate configurations.
- Never include instance-specific properties of a resource.

### Structure of an Insight

Each insight must contain three parts: an observed pattern, the reasoning behind it, and a planning implication.
- The reasoning must be grounded in factual information from the data. Do not make assumptions.
- The planning implication must state concrete actions or decisions for infra planning that align with the user's requirements.
- The reasoning must clearly connect the observed pattern to the planning implication.

### Filtering

Use the following areas as a guide when deciding which resource properties are meaningful:
- Region
- Resource pairing
- Security posture
- Cost
- Naming and tagging conventions
- Azure policies

# Output

Return a JSON object with an "insights" key containing an array of insight strings.

```json
{
  "insights": [
    "Insight 1",
    "Insight 2",
    "Insight 3"
  ]
}
```

Each insight must be a single sentence with this structure: "[observed pattern]: [reasoning] [planning implication]"."""


sub_count = len({r.get("subscriptionId") for r in resources if r.get("subscriptionId")})
rg_count = len({
    (r.get("subscriptionId"), (r.get("resourceGroup") or "").lower())
    for r in resources
    if r.get("subscriptionId") and r.get("resourceGroup")
})


def build_baseline_payload(query: str) -> str:
    return json.dumps({"userQuery": query, "resourceContext": {"subscriptionCount": sub_count, "resourceGroupCount": rg_count, "resourceTypes": resource_type_aggregations}}, ensure_ascii=False)


def build_mfi_payload(query: str) -> str:
    return json.dumps({
        "userQuery": query,
        "resourceContext": {"subscriptionCount": sub_count, "resourceGroupCount": rg_count, "resourceTypes": resource_type_aggregations},
        "tenantPatterns": {
            "transactionUnit": "resourceGroup",
            "algorithm": "fpgrowth",
            "minSupport": round(min_support, 4),
            "reliabilityBuckets": {"high": ">=30 RGs", "medium": ">=10 RGs", "low": ">=5 RGs", "exploratory": f">={MIN_ABSOLUTE_COUNT} RGs"},
            "maximalFrequentItemsets": mfis,
        },
    }, ensure_ascii=False)


def generate_insights(payload_json: str) -> list[str]:
    response = aoai.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[{"role": "system", "content": INSIGHTS_PROMPT}, {"role": "user", "content": payload_json}],
        response_format={"type": "json_object"},
    )
    raw_text = response.choices[0].message.content or "{}"
    parsed = json.loads(raw_text)
    if isinstance(parsed, list):
        return [str(x) for x in parsed]
    if isinstance(parsed, dict):
        for key in ("insights", "Insights", "items", "results"):
            if key in parsed and isinstance(parsed[key], list):
                return [str(x) for x in parsed[key]]
        for v in parsed.values():
            if isinstance(v, list) and v and isinstance(v[0], str):
                return [str(x) for x in v]
    return []


# Show payload sizes
baseline_size = len(build_baseline_payload(PROMPTS[0]))
mfi_size = len(build_mfi_payload(PROMPTS[0]))
print(f"Baseline payload: {baseline_size:,} chars")
print(f"MFI payload: {mfi_size:,} chars (+{mfi_size - baseline_size:,} chars from MFIs)")


## Generate Insights for All Prompts

In [ ]:
all_results: dict[int, dict] = {}

for i, prompt in enumerate(PROMPTS):
    print(f"\n{'='*80}")
    print(f"Prompt {i+1}: {prompt[:80]}...")
    print(f"{'='*80}")

    print(f"  Generating baseline insights...")
    baseline_payload = build_baseline_payload(prompt)
    baseline_insights = generate_insights(baseline_payload)
    print(f"  Baseline: {len(baseline_insights)} insights")

    time.sleep(2)

    print(f"  Generating MFI insights...")
    mfi_payload = build_mfi_payload(prompt)
    mfi_insights = generate_insights(mfi_payload)
    print(f"  MFI: {len(mfi_insights)} insights")

    all_results[i] = {
        "prompt": prompt,
        "baseline": baseline_insights,
        "mfi": mfi_insights,
    }

    (RESULTS_DIR / f"prompt_{i+1}_baseline.json").write_text(json.dumps(baseline_insights, indent=2, ensure_ascii=False), encoding="utf-8")
    (RESULTS_DIR / f"prompt_{i+1}_mfi.json").write_text(json.dumps(mfi_insights, indent=2, ensure_ascii=False), encoding="utf-8")

    time.sleep(2)

print(f"\nGenerated insights for {len(all_results)} prompts.")


## Pairwise Evaluation

For each prompt, compare baseline vs MFI insights in both orderings.

In [ ]:
EVAL_TEMPLATE = """Compare the two sets of insights generated. Do not try to figure out how they are generated, simply evaluate them at face value.

Evaluation criteria:
- Does each insight identify an actual architecture point (not generic advice)?
- Does each insight provide a valid planning implication?
- Is each insight grounded in facts (not guesses)?

For each set, note:
1. How many insights are architecturally substantive
2. How many provide actionable planning guidance
3. Any unique insights one set has that the other misses

Then declare a winner (Set A or Set B) or a draw, with a brief justification.

User prompt: "{prompt}"

Insight set A:
{set_a}

Insight set B:
{set_b}

Respond with JSON: {{"winner": "A" or "B" or "draw", "justification": "...", "unique_to_a": [...], "unique_to_b": [...]}}"""


def format_insights(insights: list[str]) -> str:
    return "\n".join(f"{i+1}. {ins}" for i, ins in enumerate(insights))


def evaluate_pair(prompt: str, set_a: list[str], set_b: list[str]) -> dict:
    eval_prompt = EVAL_TEMPLATE.format(
        prompt=prompt,
        set_a=format_insights(set_a),
        set_b=format_insights(set_b),
    )
    response = aoai.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[{"role": "user", "content": eval_prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content or "{}")


evaluations = []

for i, result in all_results.items():
    prompt = result["prompt"]
    baseline = result["baseline"]
    mfi = result["mfi"]
    short = prompt[:60] + "..."

    print(f"\nPrompt {i+1} \u2014 ordering 1 (A=baseline, B=mfi)...")
    eval1 = evaluate_pair(prompt, baseline, mfi)
    winner1 = {"A": "baseline", "B": "mfi"}.get(eval1.get("winner", ""), "draw")
    print(f"  Winner: {winner1}")

    time.sleep(2)

    print(f"Prompt {i+1} \u2014 ordering 2 (A=mfi, B=baseline)...")
    eval2 = evaluate_pair(prompt, mfi, baseline)
    winner2 = {"A": "mfi", "B": "baseline"}.get(eval2.get("winner", ""), "draw")
    print(f"  Winner: {winner2}")

    consistent = winner1 == winner2
    evaluations.append({
        "prompt_index": i + 1,
        "prompt_short": short,
        "ordering1": {"a": "baseline", "b": "mfi", "raw": eval1, "mapped_winner": winner1},
        "ordering2": {"a": "mfi", "b": "baseline", "raw": eval2, "mapped_winner": winner2},
        "consistent": consistent,
        "final_winner": winner1 if consistent else "inconclusive",
    })

    time.sleep(2)

print(f"\nCompleted {len(evaluations)} prompt evaluations.")


## Aggregate & Save Results

In [ ]:
wins = {"baseline": 0, "mfi": 0, "draw": 0, "inconclusive": 0}
for ev in evaluations:
    wins[ev["final_winner"]] += 1

summary = {
    "total_prompts": len(PROMPTS),
    "wins": wins,
    "consistency": sum(1 for e in evaluations if e["consistent"]),
    "per_prompt": [
        {
            "prompt_index": e["prompt_index"],
            "prompt": e["prompt_short"],
            "final_winner": e["final_winner"],
            "consistent": e["consistent"],
            "ordering1_justification": e["ordering1"]["raw"].get("justification", ""),
            "ordering2_justification": e["ordering2"]["raw"].get("justification", ""),
        }
        for e in evaluations
    ],
}

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"MFI wins:      {wins['mfi']}")
print(f"Baseline wins: {wins['baseline']}")
print(f"Draws:         {wins['draw']}")
print(f"Inconclusive:  {wins['inconclusive']}")
print(f"Consistency:   {summary['consistency']}/{len(evaluations)} verdicts agreed across orderings")
print()

for e in evaluations:
    print(f"  Prompt {e['prompt_index']}: {e['final_winner']:>12}  (consistent={e['consistent']})")
    print(f"    {e['prompt_short']}")

(RESULTS_DIR / "evaluations.json").write_text(json.dumps(evaluations, indent=2, ensure_ascii=False), encoding="utf-8")
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved evaluations.json and summary.json to {RESULTS_DIR}/")
